# 02 Qwen3-TTS ModelScope 动手测试

这份 notebook 是给你自己动手跑的，不是纯理论笔记。默认流程会用 ModelScope 下载 `Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice`，加载模型，合成一段中文语音，保存 WAV，并计算 RTF。

默认只跑 1.7B CustomVoice，因为官方已发布这个版本，效果比 0.6B 更适合作为主测试模型。VoiceDesign、VoiceClone、Tokenizer encode/decode 都在后面做成开关单元，需要时再打开。

## 0. 安装依赖

在 ModelScope Notebook 或本地新环境里先运行这一格。`qwen-tts` 会安装 Qwen3-TTS 的官方 Python 接口。

In [ ]:
%pip install -U modelscope qwen-tts soundfile

## 1. 导入依赖并选择设备

有 CUDA GPU 时用 `cuda:0` 和 `bfloat16`。没有 GPU 时会退回 CPU，但大模型在 CPU 上会很慢，只适合检查流程。

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import soundfile as sf
import torch
from IPython.display import Audio, display


def pick_runtime() -> tuple[str, torch.dtype]:
    if torch.cuda.is_available():
        return "cuda:0", torch.bfloat16
    return "cpu", torch.float32


DEVICE_MAP, DTYPE = pick_runtime()
OUTPUT_DIR = Path("speech/outputs")
MODEL_CACHE_DIR = Path("speech/modelscope_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print({"device_map": DEVICE_MAP, "dtype": str(DTYPE), "cuda": torch.cuda.is_available()})

## 2. 配置要测试的模型

默认先用 1.7B CustomVoice。这个模型支持官方内置音色和指令控制，最适合直接测试 Qwen3-TTS 的主能力。如果显存不够，再把下面的模型 ID 改回 `Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice`。

In [ ]:
CUSTOM_VOICE_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
TOKENIZER_MODEL_ID = "Qwen/Qwen3-TTS-Tokenizer-12Hz"
VOICE_DESIGN_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
VOICE_CLONE_MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

SYNTH_TEXT = "今天我们从输入到输出，动手测试 Qwen3-TTS 的语音合成流程。"
SYNTH_LANGUAGE = "Chinese"
SYNTH_SPEAKER = "Vivian"

# 如果你的 GPU 支持 FlashAttention 2，可以改成 True。
USE_FLASH_ATTENTION = False

print("custom voice model:", CUSTOM_VOICE_MODEL_ID)
print("text:", SYNTH_TEXT)

## 3. 用 ModelScope 下载模型

`snapshot_download` 会把模型下载到 ModelScope 缓存目录，并返回本地路径。后面 `Qwen3TTSModel.from_pretrained()` 直接从这个本地路径加载。

In [ ]:
from modelscope import snapshot_download


def download_from_modelscope(model_id: str) -> str:
    print(f"downloading or reusing: {model_id}")
    model_dir = snapshot_download(model_id, cache_dir=str(MODEL_CACHE_DIR))
    print("local model dir:", model_dir)
    return model_dir


custom_voice_dir = download_from_modelscope(CUSTOM_VOICE_MODEL_ID)

## 4. 先看一下下载下来的模型文件

这一步不加载大模型，只看目录里有哪些配置和权重文件。真实项目排错时很有用。

In [ ]:
def show_model_files(model_dir: str, max_items: int = 80) -> None:
    root = Path(model_dir)
    files = sorted(path for path in root.rglob("*") if path.is_file())
    print("file count:", len(files))
    for path in files[:max_items]:
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"{path.relative_to(root)}  {size_mb:.2f} MB")
    if len(files) > max_items:
        print(f"... {len(files) - max_items} more files")


show_model_files(custom_voice_dir)

## 5. 加载 Qwen3-TTS 模型

这里真正把模型加载进内存 / 显存。如果报显存不足，先换更小 batch、关闭其他进程，或者只在 ModelScope GPU Notebook 中跑。

In [ ]:
from qwen_tts import Qwen3TTSModel


load_kwargs = {
    "device_map": DEVICE_MAP,
    "dtype": DTYPE,
}
if USE_FLASH_ATTENTION:
    load_kwargs["attn_implementation"] = "flash_attention_2"

start = time.perf_counter()
model = Qwen3TTSModel.from_pretrained(custom_voice_dir, **load_kwargs)
load_elapsed = time.perf_counter() - start

print(f"loaded in {load_elapsed:.2f}s")
if hasattr(model, "get_supported_speakers"):
    print("supported speakers:", model.get_supported_speakers())
if hasattr(model, "get_supported_languages"):
    print("supported languages:", model.get_supported_languages())

## 6. 生成第一段语音

这一步就是完整链路：文本 + 语言 + speaker -> speech tokens -> waveform -> WAV 文件。

In [ ]:
start = time.perf_counter()
wavs, sr = model.generate_custom_voice(
    text=SYNTH_TEXT,
    language=SYNTH_LANGUAGE,
    speaker=SYNTH_SPEAKER,
)
elapsed = time.perf_counter() - start

output_path = OUTPUT_DIR / "qwen3_tts_custom_voice.wav"
sf.write(output_path, wavs[0], sr)

duration = len(wavs[0]) / sr
rtf = elapsed / duration if duration else float("inf")

print("saved:", output_path)
print({"sample_rate": sr, "duration_seconds": duration, "elapsed_seconds": elapsed, "rtf": rtf})
display(Audio(str(output_path)))

## 7. 改文本、speaker、语言再试一次

你可以只改下面三个变量。注意 speaker 的效果和语言相关，官方建议优先使用 speaker 的母语。

In [ ]:
TEST_TEXT = "换一个说话人，再测试一遍语音合成。"
TEST_LANGUAGE = "Chinese"
TEST_SPEAKER = "Serena"

start = time.perf_counter()
test_wavs, test_sr = model.generate_custom_voice(
    text=TEST_TEXT,
    language=TEST_LANGUAGE,
    speaker=TEST_SPEAKER,
)
elapsed = time.perf_counter() - start

test_output_path = OUTPUT_DIR / "qwen3_tts_custom_voice_changed.wav"
sf.write(test_output_path, test_wavs[0], test_sr)

duration = len(test_wavs[0]) / test_sr
print("saved:", test_output_path)
print({"sample_rate": test_sr, "duration_seconds": duration, "elapsed_seconds": elapsed, "rtf": elapsed / duration})
display(Audio(str(test_output_path)))

## 8. 可选：VoiceDesign，用自然语言描述声音

这一节会下载 `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`，模型更大。要跑就把 `RUN_VOICE_DESIGN` 改成 `True`。

In [ ]:
RUN_VOICE_DESIGN = False

if RUN_VOICE_DESIGN:
    voice_design_dir = download_from_modelscope(VOICE_DESIGN_MODEL_ID)
    design_model = Qwen3TTSModel.from_pretrained(voice_design_dir, **load_kwargs)

    design_wavs, design_sr = design_model.generate_voice_design(
        text="这是一段用于测试语音设计的中文合成。",
        language="Chinese",
        instruct="年轻女性，声音温柔自然，语速稍慢，情绪稳定。",
    )

    design_output_path = OUTPUT_DIR / "qwen3_tts_voice_design.wav"
    sf.write(design_output_path, design_wavs[0], design_sr)
    print("saved:", design_output_path)
    display(Audio(str(design_output_path)))
else:
    print("skip VoiceDesign. Set RUN_VOICE_DESIGN = True to run this cell.")

## 9. 可选：VoiceClone，用参考音频克隆音色

准备一段干净、短的 `reference.wav`，再把 `REFERENCE_AUDIO_PATH` 和 `REFERENCE_TEXT` 改成自己的内容。`ref_text` 要尽量和参考音频一字不差。

In [ ]:
RUN_VOICE_CLONE = False
REFERENCE_AUDIO_PATH = "speech/outputs/qwen3_tts_custom_voice.wav"
REFERENCE_TEXT = SYNTH_TEXT

if RUN_VOICE_CLONE:
    if not Path(REFERENCE_AUDIO_PATH).exists():
        raise FileNotFoundError(f"reference audio not found: {REFERENCE_AUDIO_PATH}")

    clone_dir = download_from_modelscope(VOICE_CLONE_MODEL_ID)
    clone_model = Qwen3TTSModel.from_pretrained(clone_dir, **load_kwargs)

    clone_wavs, clone_sr = clone_model.generate_voice_clone(
        text="这句话会尽量使用参考音频里的音色来说。",
        language="Chinese",
        ref_audio=REFERENCE_AUDIO_PATH,
        ref_text=REFERENCE_TEXT,
    )

    clone_output_path = OUTPUT_DIR / "qwen3_tts_voice_clone.wav"
    sf.write(clone_output_path, clone_wavs[0], clone_sr)
    print("saved:", clone_output_path)
    display(Audio(str(clone_output_path)))
else:
    print("skip VoiceClone. Set RUN_VOICE_CLONE = True to run this cell.")

## 10. 可选：单独测试 speech tokenizer encode / decode

这一节不需要输入文本，只测试：音频 -> speech tokens -> 重建音频。它能帮助你理解 Qwen3-TTS 中间的离散语音 token。

In [ ]:
RUN_TOKENIZER_TEST = False
TOKENIZER_INPUT_AUDIO = "speech/outputs/qwen3_tts_custom_voice.wav"

if RUN_TOKENIZER_TEST:
    if not Path(TOKENIZER_INPUT_AUDIO).exists():
        raise FileNotFoundError(f"tokenizer input audio not found: {TOKENIZER_INPUT_AUDIO}")

    from qwen_tts import Qwen3TTSTokenizer

    tokenizer_dir = download_from_modelscope(TOKENIZER_MODEL_ID)
    tts_tokenizer = Qwen3TTSTokenizer.from_pretrained(tokenizer_dir, device_map=DEVICE_MAP)

    codes = tts_tokenizer.encode(TOKENIZER_INPUT_AUDIO)
    recon_wavs, recon_sr = tts_tokenizer.decode(codes)

    recon_output_path = OUTPUT_DIR / "qwen3_tts_tokenizer_reconstruct.wav"
    sf.write(recon_output_path, recon_wavs[0], recon_sr)
    print("saved:", recon_output_path)
    print("codes type:", type(codes))
    display(Audio(str(recon_output_path)))
else:
    print("skip tokenizer test. Set RUN_TOKENIZER_TEST = True to run this cell.")

## 11. 对照：你刚才跑通了哪几步

如果第 6 节成功生成 WAV，就说明下面这条完整链路已经跑通了。

In [ ]:
steps = [
    "1. 输入 text / language / speaker",
    "2. qwen-tts 整理模型请求",
    "3. Qwen tokenizer 把文本转成 token",
    "4. Qwen3-TTS 语言模型预测 speech tokens",
    "5. Qwen-TTS-Tokenizer-12Hz / Code2Wav 解码 waveform",
    "6. soundfile 写入 WAV",
    "7. 用 Audio 播放结果，并用 RTF 看速度",
]

print("\n".join(steps))
print("\noutputs:")
for path in sorted(OUTPUT_DIR.glob("qwen3_tts_*.wav")):
    print(path)